# 01 — RoBERTa Document Retrieval
TF-IDF Top-100을 HoVer Train hard negative로 fine-tune한 RoBERTa가 재정렬합니다. 모델 선택은 `hpqa_id` 그룹 기반 Train 내부 Validation만 사용합니다. 기본은 Dev 2-Hop 1,126건이며 이후 `HOPS='2,3,4'`로 재실행하면 체크포인트에 이어 붙입니다.

In [13]:
from pathlib import Path
import os, subprocess, sys
DRIVE_CANDIDATE = Path('/content/drive/MyDrive/hover-project')
DRIVE_PROJECT = DRIVE_CANDIDATE if DRIVE_CANDIDATE.is_dir() else None
roots = ([Path(os.environ['HOVER_E2E_ROOT']).expanduser()] if os.environ.get('HOVER_E2E_ROOT') else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/e2e_v2'), Path('/content/고준서/e2e_v2'), Path('/content/Fact-Verfication')] + ([DRIVE_PROJECT] if DRIVE_PROJECT else [])
candidates = [candidate for root in roots for candidate in (root, root / 'e2e_v2', root / '고준서/e2e_v2')]
E2E_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file() and (p / 'src/hover_e2e').is_dir()), None)
if E2E_ROOT is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_PROJECT = Path('/content/drive/MyDrive/hover-project')
        mounted_candidates = [DRIVE_PROJECT, DRIVE_PROJECT / 'e2e_v2', DRIVE_PROJECT / '고준서/e2e_v2']
        E2E_ROOT = next((p.resolve() for p in mounted_candidates if (p / 'pyproject.toml').is_file() and (p / 'src/hover_e2e').is_dir()), None)
    except ImportError:
        pass
if E2E_ROOT is None: raise FileNotFoundError('현재 커널과 마운트된 Drive에서 e2e_v2 소스를 찾지 못했습니다. /content/e2e_v2 또는 MyDrive/hover-project/고준서/e2e_v2에 폴더가 필요합니다.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{E2E_ROOT}[gpu,test]'])
sys.path.insert(0, str(E2E_ROOT / 'src')) if str(E2E_ROOT / 'src') not in sys.path else None
from hover_e2e.colab_runtime import prepare_runtime
RUN_NAME = os.environ.get('HOVER_RUN_NAME', 'roberta_e2e_v2')
HOPS = os.environ.get('HOVER_HOPS', '2')
CTX = prepare_runtime(E2E_ROOT, drive_project=DRIVE_PROJECT, run_name=RUN_NAME)
CONFIG = E2E_ROOT / 'configs/base.json'
# 01 성능 안전 패치: Train inference만 config.rank_k(기본 20) 사용, Dev는 Top-100 유지.
# Colab의 /content/e2e_v2는 이전 00_preflight가 복구한 코드일 수 있어 여기서 직접 반영.
stage_file = E2E_ROOT / 'src/hover_e2e/stages.py'
original = stage_file.read_text(encoding='utf-8')
old = "ranks[uid][:int(config[\"tfidf_k\"])],"
new = "ranks[uid][:int(config['rank_k'] if split == 'train' else config['tfidf_k'])],"
if old in original:
    assert original.count(old) == 1, 'Unexpected code layout, aborting safe patch'
    stage_file.write_text(original.replace(old, new), encoding='utf-8')
    print('[patch] Train scoring Top-20, Dev scoring Top-100 applied', flush=True)
elif new in original:
    print('[patch] fast Train scoring already applied', flush=True)
else:
    raise RuntimeError('stages.py changed: manual inspection required, not silently patching')
print('[check] stage source:', stage_file, flush=True)
import torch
assert torch.cuda.is_available(), 'GPU 런타임으로 변경하세요.'
print(torch.cuda.get_device_name(0), '| hops', HOPS)


reuse repository-local HoVer data: /content/hover_data
[patch] Train scoring Top-20, Dev scoring Top-100 applied
[check] stage source: /content/e2e_v2/src/hover_e2e/stages.py
NVIDIA L4 | hops 2


In [14]:
from pathlib import Path
import subprocess, sys, time

command = [
    sys.executable, '-u', '-m', 'hover_e2e.cli', 'retrieval',
    '--config', str(CONFIG), '--data-root', str(CTX.data_root),
    '--output-root', str(CTX.output_root), '--run-name', RUN_NAME,
    '--hops', HOPS,
]
log_path = CTX.output_root / RUN_NAME / '01_retrieval_live.log'
log_path.parent.mkdir(parents=True, exist_ok=True)
print('COMMAND:', ' '.join(command), flush=True)
print('LIVE LOG:', log_path, flush=True)
print('NOTE: 로드/학습/Train 추론 전체가 포함된 장시간 GPU 작업입니다.', flush=True)
start = time.perf_counter()
with log_path.open('a', encoding='utf-8') as logfile:
    logfile.write('\n===== run started =====\n')
    logfile.flush()
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            logfile.write(line)
            logfile.flush()
    except KeyboardInterrupt:
        process.terminate()
        raise
    finally:
        ret = process.wait()
        logfile.write(f'\n===== exit={ret}, elapsed={time.perf_counter()-start:.1f}s =====\n')
        logfile.flush()
if ret:
    raise subprocess.CalledProcessError(ret, command)
print('Retrieval complete. 다음: 02_multihop.ipynb', flush=True)


COMMAND: /usr/bin/python3 -u -m hover_e2e.cli retrieval --config /content/e2e_v2/configs/base.json --data-root /content/hover_data --output-root /content/drive/MyDrive/hover-project/고준서/e2e_v2_runs --run-name roberta_e2e_v2 --hops 2
LIVE LOG: /content/drive/MyDrive/hover-project/고준서/e2e_v2_runs/roberta_e2e_v2/01_retrieval_live.log
NOTE: 로드/학습/Train 추론 전체가 포함된 장시간 GPU 작업입니다.
I0000 00:00:1793068656.625014   23335 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793068656.696131   23335 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793068659.0959